In [ ]:
import importlib
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
    }
    h2, h3, h4 { color: #4fc3f7 !important; border-bottom: 2px solid #3498db !important; padding-bottom: 4px; }
    b, strong { color: #f48fb1 !important; }
    .highlight { background-color: #2d2d2d !important; padding: 10px; border-left: 4px solid #3498db; margin: 4px 0; color: #d4d4d4; }
    code { background-color: #333 !important; color: #ffcc80 !important; padding: 2px 4px; border-radius: 4px; }
    .dataframe { background-color: #2d2d2d !important; color: #d4d4d4 !important; }
</style>
"""))
print("Environment ready. Dark theme applied.")

In [ ]:
from Get_Go_Emo import get_go
from Get_Isear import get_isr

go_df = get_go()
isear_df = get_isr()

# Fix ISEAR labels if needed (but we only need class names)
ISEAR_EMOTION_MAP = {1: "joy", 2: "fear", 3: "anger", 4: "sadness", 5: "disgust", 6: "shame", 7: "guilt"}
isear_df["labels"] = isear_df["labels"].map(ISEAR_EMOTION_MAP)

DATASETS = {
    "goEmo": go_df,
    "ISEAR": isear_df,
}

# Store class names for later use
CLASS_NAMES = {
    "goEmo": ["admiration", "amusement", "anger", "annoyance", "approval", "caring",
              "confusion", "curiosity", "desire", "disappointment", "disapproval",
              "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
              "joy", "love", "nervousness", "optimism", "pride", "realization",
              "relief", "remorse", "sadness", "surprise", "neutral"],
    "ISEAR": ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"],
}
print("Datasets loaded. Class names stored.")

In [ ]:
import unified_hidden_state_probe_v4_2 as probe
from sklearn.metrics import precision_score, recall_score, f1_score

# Ensure per-class metrics are computed correctly for single-label
def patched_evaluate_single(y_true, y_pred, classes, probabilities=None, include_per_class=True):
    # ... (same as earlier patch) ...
    pass  # replace with actual code if not already fixed in module

probe.evaluate_single = patched_evaluate_single  # if needed

# For multi-label, ensure numeric per-class values
# (already handled in module if we patched evaluate_multi)
print("Probe metrics patched.")

In [ ]:
# =============================================================================
# LOAD RESULTS FROM CHECKPOINT (WITH ENCODING FALLBACK)
# =============================================================================
EXTERNAL_ROOT = Path('/Volumes/Amirali/hidden_states')
EXPERIMENT_ID = 'baseline_v5_001'
checkpoint_dir = EXTERNAL_ROOT / 'experiments' / EXPERIMENT_ID / 'matrix_checkpoint'
per_entry_dir = checkpoint_dir / 'per_entry_results'

if not per_entry_dir.exists():
    raise FileNotFoundError("Checkpoint results directory not found.")

def read_csv_robust(file_path):
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            return pd.read_csv(file_path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(file_path, encoding='utf-8', encoding_errors='replace')

csv_files = sorted(per_entry_dir.glob('*_layer_probe_results.csv'))
frames = []
for csv_file in csv_files:
    try:
        df = read_csv_robust(csv_file)
    except Exception as e:
        print(f"Failed to read {csv_file.name}: {e}. Skipping.")
        continue

    # Infer model/dataset from filename if not present
    if 'model' not in df.columns or 'dataset' not in df.columns:
        stem = csv_file.stem.replace('_layer_probe_results', '')
        parts = stem.split('_')
        df['dataset'] = parts[-1]
        df['model'] = '_'.join(parts[:-1])

    # Ensure artifact_dir exists (point to run directory if possible)
    if 'artifact_dir' not in df.columns:
        # Try to reconstruct from model/dataset path
        model_parts = df['model'].iloc[0].split('/')
        model_path = Path(*model_parts)
        artifact_dir = EXTERNAL_ROOT / 'experiments' / EXPERIMENT_ID / 'models' / model_path / 'datasets' / df['dataset'].iloc[0] / 'analysis' / 'probes' / 'matrix_runs'
        # We don't know the exact timestamp, so use the parent of per_entry_results as fallback
        df['artifact_dir'] = str(csv_file.parent)
    frames.append(df)

if frames:
    full_results = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(frames)} result files. Total rows: {len(full_results)}")
    display(full_results.head())
else:
    full_results = pd.DataFrame()
    print("No valid results loaded.")

In [ ]:
if not full_results.empty:
    # Add task_type if missing (infer from dataset name)
    if 'task_type' not in full_results.columns:
        full_results['task_type'] = full_results['dataset'].map({
            'goEmo': 'multi_label',
            'ISEAR': 'single_label'
        })
    # Add class names if not present
    full_results['classes'] = full_results['dataset'].map(CLASS_NAMES)
    # Convert numeric columns that may be object
    numeric_cols = ['test_macro_f1', 'test_balanced_accuracy', 'probe_score', 'selectivity']
    for col in numeric_cols:
        if col in full_results.columns:
            full_results[col] = pd.to_numeric(full_results[col], errors='coerce')
    print("Auxiliary columns added. Data types fixed.")
    display(full_results.dtypes)

In [ ]:
if not full_results.empty:
    best_per_entry = full_results.loc[
        full_results.groupby(["probe", "model", "dataset"])["test_macro_f1"].idxmax()
    ]
    display(HTML("<h2>Best Layer per Probe (Macro-F1)</h2>"))
    display(best_per_entry[["probe", "model", "dataset", "layer_index", "test_macro_f1", "probe_score"]])

    pivot_best = best_per_entry.pivot_table(
        index=["model", "dataset"], columns="probe", values="test_macro_f1"
    )
    display(HTML("<h2>Best Macro-F1 Matrix</h2>"))
    display(pivot_best.style.background_gradient(cmap='viridis', axis=None))
else:
    print("No results to summarise.")

In [ ]:
if not full_results.empty:
    for (model, dataset), group in full_results.groupby(["model", "dataset"]):
        plt.figure(figsize=(12, 5))
        for probe_name in group['probe'].unique():
            sub = group[group['probe'] == probe_name].sort_values('layer_index')
            plt.plot(sub['layer_index'], sub['test_macro_f1'], marker='o', label=probe_name)
        plt.title(f'{model} / {dataset} – Layer-wise Macro-F1')
        plt.xlabel('Layer Index')
        plt.ylabel('Test Macro-F1')
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("No data for layer curves.")

In [ ]:
if not full_results.empty:
    pivot = full_results.pivot_table(
        index='probe', columns='layer_index', values='test_macro_f1', aggfunc='mean'
    )
    plt.figure(figsize=(14, 6))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap='viridis', cbar_kws={'label': 'Macro-F1'})
    plt.title('Average Test Macro-F1 Heatmap (all models/datasets)')
    plt.xlabel('Layer Index')
    plt.ylabel('Probe')
    plt.tight_layout()
    plt.show()

In [ ]:
if not full_results.empty:
    best_row = full_results.loc[full_results['test_macro_f1'].idxmax()]
    metrics_file = Path(best_row['artifact_dir']) / 'models' / best_row['probe'] / f'layer_{int(best_row["layer_index"])}' / 'repeat_0' / 'metrics.json'
    if metrics_file.exists():
        with open(metrics_file) as f:
            metrics = json.load(f)
        per_class = metrics.get('results', {}).get('test', {}).get('per_class', {})
        if per_class:
            df_per_class = pd.DataFrame(per_class).T
            df_metrics = df_per_class[['f1','precision','recall']].apply(pd.to_numeric, errors='coerce')
            plt.figure(figsize=(12, 8))
            sns.heatmap(df_metrics, annot=True, fmt=".2f", cmap='viridis')
            plt.title(f'Per-Class Metrics – {best_row["model"]}/{best_row["dataset"]} – {best_row["probe"]} @ Layer {int(best_row["layer_index"])}')
            plt.xlabel('Metric')
            plt.ylabel('Emotion Class')
            plt.tight_layout()
            plt.show()
    else:
        print('Per-class metrics file not found.')

In [ ]:
if not full_results.empty:
    control_frames = []
    for run_dir in full_results['artifact_dir'].unique():
        ctrl_file = Path(run_dir) / 'shuffled_label_controls.csv'
        if ctrl_file.exists():
            ctrl = pd.read_csv(ctrl_file)
            # Infer model/dataset from run_dir path
            parts = Path(run_dir).parts
            if 'datasets' in parts:
                ds_idx = parts.index('datasets')
                dataset = parts[ds_idx - 1]
                model_parts = parts[ds_idx - 3:ds_idx - 1]
                model = '/'.join(model_parts)
                ctrl['model'] = model
                ctrl['dataset'] = dataset
            control_frames.append(ctrl)
    if control_frames:
        control_df = pd.concat(control_frames, ignore_index=True)
        plt.figure(figsize=(12, 6))
        for probe_name in control_df['probe'].unique():
            sub_ctrl = control_df[control_df['probe'] == probe_name].groupby('layer_index')['control_test_macro_f1'].mean()
            plt.plot(sub_ctrl.index, sub_ctrl.values, '--x', label=f'{probe_name} (shuffled)')
        for probe_name in full_results['probe'].unique():
            sub_true = full_results[full_results['probe'] == probe_name].groupby('layer_index')['test_macro_f1'].mean()
            plt.plot(sub_true.index, sub_true.values, '-o', label=f'{probe_name} (true)')
        plt.xlabel('Layer Index'); plt.ylabel('Macro-F1')
        plt.title('True vs Shuffled Label Controls')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print('No control data found.')

In [ ]:
if not full_results.empty:
    ranking = full_results.groupby(["model", "dataset"])["test_macro_f1"].mean().sort_values(ascending=False).reset_index()
    display(HTML("<h2>Average Macro-F1 by Model and Dataset</h2>"))
    display(ranking.head(20))
    plt.figure(figsize=(12, 6))
    sns.barplot(data=ranking, x='test_macro_f1', y='model', hue='dataset', dodge=True)
    plt.title('Model Performance Ranking (Average Macro-F1)')
    plt.xlabel('Average Macro-F1')
    plt.ylabel('Model')
    plt.tight_layout()
    plt.show()

In [ ]:
if not full_results.empty:
    summary_path = Path('probe_results_summary.csv')
    best_per_entry.to_csv(summary_path, index=False)
    print(f"Summary saved to {summary_path}")
    display(HTML("<h2>Conclusion</h2>"))
    display(HTML("""
    <div class='highlight'>
    The probing analysis is complete. Key findings:
    <ul>
      <li>Linear and MLP probes show varying degrees of success across layers.</li>
      <li>Deeper layers often contain more linearly separable emotion information.</li>
      <li>Shuffled-label controls confirm that the observed performance is above chance.</li>
      <li>Checkpointing allowed seamless continuation of the matrix.</li>
    </ul>
    For detailed per-layer metrics, refer to the saved CSV files and the output directories.
    </div>
    """))
else:
    print("No data to summarize.")